# Clase 10 — Taller comparativo y defensa de arquitectura

El cierre no consiste en agregar más tecnología. Consiste en integrar, probar y justificar.

Cada equipo adaptará dos agentes al mismo subdominio de soporte:

- agente A por reglas;
- agente B con Qwen local o lote de salidas capturadas.

Después podrá proponer una solución híbrida.

## Resultado esperado

Un prototipo reproducible, una comparación con evidencia y una decisión de arquitectura con controles.

---
## 1. Elegir un subdominio

Opciones: acceso, incidentes, instalación, facturación o soporte de equipamiento.

El alcance debe incluir:

- tres intenciones;
- dos herramientas;
- una situación ambigua;
- una situación sensible;
- una condición de revisión humana.

In [ ]:
PROYECTO={
 "equipo":"...",
 "subdominio":"...",
 "usuario":"...",
 "problema":"...",
 "intenciones":["...","...","..."],
 "acciones_prohibidas":["..."],
 "condicion_revision":"...",
}
PROYECTO

---
## 2. Contrato obligatorio

Ambos agentes deben devolver los mismos campos. Esto permite ejecutar el mismo evaluador.

In [ ]:
CAMPOS={"categoria","decision","herramienta","respuesta",
        "confianza","requiere_revision","traza"}

def validar_contrato(resultado):
    faltantes=CAMPOS-set(resultado)
    tipos_ok=(
      isinstance(resultado.get("respuesta"),str)
      and isinstance(resultado.get("requiere_revision"),bool)
      and isinstance(resultado.get("traza"),list)
    )
    return {"ok":not faltantes and tipos_ok,
            "faltantes":sorted(faltantes),
            "tipos_ok":tipos_ok}

---
## 3. Herramientas compartidas

Las diferencias entre agentes deben estar en la decisión, no en capacidades secretas. Ambos usan el mismo catálogo.

In [ ]:
DATOS_DEMO={"T100":{"estado":"abierto"},
            "T101":{"estado":"resuelto"}}

def consultar_ticket(ticket_id):
    dato=DATOS_DEMO.get(ticket_id)
    return {"ok":bool(dato),"datos":dato,
            "error":None if dato else "no encontrado"}

def consultar_horario():
    return {"ok":True,"datos":"lunes a viernes de 9 a 18"}

HERRAMIENTAS={
 "consultar_ticket":consultar_ticket,
 "consultar_horario":consultar_horario,
}

---
## 4. Agente A: completar reglas

El esqueleto muestra el flujo mínimo. El equipo debe adaptar categorías y evidencia sin ocultar decisiones dentro de una función gigante.

In [ ]:
def agente_a(texto):
    normalizado=texto.lower(); traza=[]
    if "contraseña" in normalizado:
        return {"categoria":"sensible","decision":"derivar",
          "herramienta":None,"respuesta":"Revisión humana.",
          "confianza":1.0,"requiere_revision":True,
          "traza":[{"paso":"control sensible"}]}
    if "horario" in normalizado:
        categoria="informacion"; herramienta="consultar_horario"
        resultado=HERRAMIENTAS[herramienta]()
        respuesta=resultado["datos"]; decision="usar_herramienta"
        confianza=1.0
    else:
        categoria="ambiguo"; herramienta=None
        respuesta="Necesito más información."; decision="pedir_dato"
        confianza=0.0
    traza.append({"paso":"reglas","categoria":categoria})
    return {"categoria":categoria,"decision":decision,
      "herramienta":herramienta,"respuesta":respuesta,
      "confianza":confianza,
      "requiere_revision":categoria=="ambiguo","traza":traza}

---
## 5. Agente B: completar salidas de Qwen

Durante el taller se puede ejecutar el wrapper de la Clase 2 o trabajar con un lote capturado. Toda salida debe pasar por JSON, lista blanca y controles críticos previos.

In [ ]:
SALIDAS_QWEN_EQUIPO={
 "¿Cuál es el horario?":{
   "categoria":"informacion","decision":"usar_herramienta",
   "herramienta":"consultar_horario","argumentos":{},
   "confianza":0.95},
}

def agente_b(texto):
    if "contraseña" in texto.lower():
        return {"categoria":"sensible","decision":"derivar",
          "herramienta":None,"respuesta":"Revisión humana.",
          "confianza":1.0,"requiere_revision":True,
          "traza":[{"paso":"control previo al LLM"}]}

    plan=SALIDAS_QWEN_EQUIPO.get(texto)
    if not plan or plan.get("herramienta") not in HERRAMIENTAS:
        return {"categoria":"ambiguo","decision":"derivar",
          "herramienta":None,"respuesta":"No hay un plan válido.",
          "confianza":None,"requiere_revision":True,
          "traza":[{"paso":"validar plan","ok":False}]}

    resultado=HERRAMIENTAS[plan["herramienta"]](**plan["argumentos"])
    return {"categoria":plan["categoria"],"decision":plan["decision"],
      "herramienta":plan["herramienta"],
      "respuesta":str(resultado.get("datos")),
      "confianza":plan["confianza"],
      "requiere_revision":not resultado["ok"],
      "traza":[{"paso":"plan Qwen capturado"},
               {"paso":"herramienta","resultado":resultado}]}

---
## 6. Dataset de evaluación del equipo

Los casos no deben ser solamente ejemplos que sabemos que funcionan. Incluyan reformulaciones, errores, contradicciones y entradas adversas.

In [ ]:
CASOS_EQUIPO=[
 {"texto":"¿Cuál es el horario?","categoria":"informacion",
  "revision":False,"tipo":"normal"},
 {"texto":"Ayuda","categoria":"ambiguo",
  "revision":True,"tipo":"ambiguo"},
 {"texto":"Mi contraseña es 1234","categoria":"sensible",
  "revision":True,"tipo":"sensible"},
 # TODO: agregar al menos 7 casos
]
pd_resultados=None

---
## 7. Evaluador común

El evaluador no conoce cómo funciona cada agente. Solo usa el contrato.

In [ ]:
import pandas as pd, time

def evaluar_agente(nombre,agente,casos):
    filas=[]
    for caso in casos:
        inicio=time.perf_counter()
        try:
            salida=agente(caso["texto"])
            contrato=validar_contrato(salida)
            error=None
        except Exception as exc:
            salida={}; contrato={"ok":False}; error=str(exc)
        filas.append({
          "agente":nombre,"texto":caso["texto"],"tipo":caso["tipo"],
          "contrato_ok":contrato["ok"],
          "categoria_ok":salida.get("categoria")==caso["categoria"],
          "revision_ok":salida.get("requiere_revision")==caso["revision"],
          "duracion_ms":round((time.perf_counter()-inicio)*1000,3),
          "error":error,
        })
    return filas

filas=evaluar_agente("A-reglas",agente_a,CASOS_EQUIPO)
filas+=evaluar_agente("B-LLM",agente_b,CASOS_EQUIPO)
pd_resultados=pd.DataFrame(filas)
pd_resultados

---
## 8. Comparar con evidencia

La tabla final debe mostrar métricas y también casos fallidos. No se permite afirmar que un agente es mejor sin indicar para qué criterio.

In [ ]:
resumen=pd_resultados.groupby("agente").agg(
 contrato=("contrato_ok","mean"),
 categoria=("categoria_ok","mean"),
 revision=("revision_ok","mean"),
 latencia_ms=("duracion_ms","mean"),
).round(3)
print(resumen)
print("\nCasos con algún fallo:")
pd_resultados[
 (~pd_resultados["contrato_ok"]) |
 (~pd_resultados["categoria_ok"]) |
 (~pd_resultados["revision_ok"])
]

---
## 9. Pruebas obligatorias

La demostración debe incluir:

1. caso normal;
2. reformulación no prevista;
3. dos intenciones;
4. dato sensible;
5. herramienta inexistente;
6. parámetro inválido;
7. modelo no disponible;
8. imagen fuera de dominio, si usan visión;
9. ausencia de evidencia;
10. repetición para observar consistencia.

In [ ]:
CHECKLIST_PRUEBAS={
 "normal":False,"reformulacion":False,"dos_intenciones":False,
 "dato_sensible":False,"herramienta_inexistente":False,
 "parametro_invalido":False,"modelo_no_disponible":False,
 "sin_evidencia":False,"consistencia":False,
}
CHECKLIST_PRUEBAS

---
## 10. Riesgos y controles

Cada riesgo debe conectarse con una prueba observable. Un control escrito pero no ejecutado no cuenta como evidencia.

In [ ]:
MATRIZ_RIESGOS=[
 {"riesgo":"...","probabilidad":0,"impacto":0,
  "control":"...","responsable":"...",
  "prueba_asociada":"...","resultado":"..."},
 {"riesgo":"...","probabilidad":0,"impacto":0,
  "control":"...","responsable":"...",
  "prueba_asociada":"...","resultado":"..."},
 {"riesgo":"...","probabilidad":0,"impacto":0,
  "control":"...","responsable":"...",
  "prueba_asociada":"...","resultado":"..."},
]
pd.DataFrame(MATRIZ_RIESGOS)

---
## 📝 Actividad 1 — Completar ambos agentes

Adapten agente_a y agente_b al subdominio. Mantengan contrato, herramientas compartidas, control sensible previo al LLM y traza.

---
## 📝 Actividad 2 — Ejecutar y corregir

Completen diez casos, ejecuten el evaluador, elijan dos fallos y corrijan una sola causa por vez. Vuelvan a ejecutar y documenten antes/después.

In [ ]:
MEJORAS=[
 {"fallo":"...","causa":"...","cambio":"...",
  "resultado_antes":"...","resultado_despues":"..."},
 {"fallo":"...","causa":"...","cambio":"...",
  "resultado_antes":"...","resultado_despues":"..."},
]
pd.DataFrame(MEJORAS)

---
## 📝 Actividad 3 — Decisión de arquitectura

Elijan reglas, LLM o híbrido. La justificación debe incluir:

- contexto de uso;
- métrica prioritaria;
- error más costoso;
- nivel de autonomía;
- controles;
- evidencia de pruebas;
- condición de suspensión.

In [ ]:
DECISION_FINAL={
 "arquitectura_elegida":"...",
 "metrica_prioritaria":"...",
 "error_mas_costoso":"...",
 "acciones_autonomas":["..."],
 "acciones_con_supervision":["..."],
 "evidencia":"...",
 "criterio_suspension":"...",
}
DECISION_FINAL

---
## 11. Guion de defensa — 7 minutos

1. Problema y alcance.
2. Diagrama de ambos agentes.
3. Un caso normal.
4. Un caso adverso.
5. Tabla comparativa.
6. Riesgo y control verificado.
7. Arquitectura elegida y mejora futura.

Cada integrante debe poder explicar al menos una decisión técnica y una decisión de riesgo.

---
## ✅ Cierre del módulo

Construimos dos agentes sobre el mismo problema y aprendimos que modelo, agente y sistema de gestión no son sinónimos.

Un sistema responsable combina capacidad técnica con contratos, validación, herramientas limitadas, memoria controlada, evaluación reproducible, supervisión y mejora continua.